# Chapter 2 — Linked Lists

A linked list is a chain of nodes, each pointing to the next (and, if doubly linked, the previous). No `O(1)` random access, but `O(1)` insertion/removal at a known node.

Techniques: the **runner** (fast/slow pointer) and **recursion**. Python has no built-in singly linked list (`collections.deque` is a doubly-linked block list), so we build our own.

In [1]:
from __future__ import annotations

from collections.abc import Iterable, Iterator
from dataclasses import dataclass


@dataclass(eq=False)  # identity equality: two nodes are equal only if they're the same object
class Node:
    val: int
    next: Node | None = None


def from_list(values: Iterable[int]) -> Node | None:
    head: Node | None = None
    tail: Node | None = None
    for v in values:
        node = Node(v)
        if tail is None:
            head = tail = node
        else:
            tail.next = node
            tail = node
    return head


def iter_nodes(head: Node | None) -> Iterator[Node]:
    while head:
        yield head
        head = head.next


def to_list(head: Node | None) -> list[int]:
    return [n.val for n in iter_nodes(head)]


class LinkedList:
    """A small singly-linked list class to show the basic operations."""

    def __init__(self, values: Iterable[int] = ()) -> None:
        self.head = from_list(values)

    def append(self, val: int) -> None:
        if not self.head:
            self.head = Node(val)
            return
        cur = self.head
        while cur.next:
            cur = cur.next
        cur.next = Node(val)

    def delete(self, val: int) -> bool:
        dummy = Node(0, self.head)
        cur = dummy
        while cur.next:
            if cur.next.val == val:
                cur.next = cur.next.next
                self.head = dummy.next
                return True
            cur = cur.next
        return False

    def __iter__(self) -> Iterator[int]:
        return (n.val for n in iter_nodes(self.head))


ll = LinkedList([1, 2, 3])
ll.append(4)
ll.delete(1)
assert list(ll) == [2, 3, 4]
print(list(ll))

[2, 3, 4]


## 2.1 Remove Dups
Remove duplicate values from an unsorted linked list. Follow-up: no temporary buffer allowed.

In [2]:
def remove_dups(head: Node | None) -> Node | None:
    """Hash set: O(N) time, O(N) space."""
    seen: set[int] = set()
    prev: Node | None = None
    cur = head
    while cur:
        if cur.val in seen:
            assert prev is not None
            prev.next = cur.next
        else:
            seen.add(cur.val)
            prev = cur
        cur = cur.next
    return head


def remove_dups_no_buffer(head: Node | None) -> Node | None:
    """Runner: O(N^2) time, O(1) space."""
    cur = head
    while cur:
        runner = cur
        while runner.next:
            if runner.next.val == cur.val:
                runner.next = runner.next.next
            else:
                runner = runner.next
        cur = cur.next
    return head


for fn in (remove_dups, remove_dups_no_buffer):
    assert to_list(fn(from_list([1, 2, 1, 3, 2, 2, 4]))) == [1, 2, 3, 4]
    assert to_list(fn(None)) == []
print("2.1 ok")

2.1 ok


## 2.2 Return Kth to Last
Find the k-th to last element (k=1 → last).

In [3]:
def kth_to_last(head: Node | None, k: int) -> Node | None:
    """Two pointers k apart. O(N) time, O(1) space."""
    lead = head
    for _ in range(k):
        if lead is None:
            return None
        lead = lead.next
    trail = head
    while lead:
        lead, trail = lead.next, trail.next  # type: ignore[union-attr]
    return trail


def kth_to_last_recursive(head: Node | None, k: int) -> Node | None:
    """Recursion returns the index-from-end on the way back up."""
    result: Node | None = None

    def helper(node: Node | None) -> int:
        nonlocal result
        if node is None:
            return 0
        idx = helper(node.next) + 1
        if idx == k:
            result = node
        return idx

    helper(head)
    return result


lst = from_list([1, 2, 3, 4, 5])
for fn in (kth_to_last, kth_to_last_recursive):
    assert fn(lst, 1).val == 5 and fn(lst, 5).val == 1 and fn(lst, 6) is None
print("2.2 ok")

2.2 ok


## 2.3 Delete Middle Node
Delete a node from the interior of a list given **only** that node. Trick: copy the next node's data over this one, then skip the next node. (Cannot work for the tail.)

In [4]:
def delete_middle_node(node: Node) -> None:
    if node.next is None:
        raise ValueError("cannot delete the last node with this technique")
    node.val = node.next.val
    node.next = node.next.next


lst = from_list(list("abcdef".encode()))
c = lst.next.next
delete_middle_node(c)
assert bytes(to_list(lst)) == b"abdef"
print("2.3 ok")

2.3 ok


## 2.4 Partition
Rearrange so all nodes less than x come before nodes ≥ x. Relative order within the partitions doesn't matter, and x may appear anywhere in the right half.

In [5]:
def partition(head: Node | None, x: int) -> Node | None:
    """Build two lists (stable), then stitch them."""
    before = before_tail = Node(0)
    after = after_tail = Node(0)
    for node in list(iter_nodes(head)):
        node.next = None
        if node.val < x:
            before_tail.next = node
            before_tail = node
        else:
            after_tail.next = node
            after_tail = node
    before_tail.next = after.next
    return before.next


def partition_head_tail(head: Node | None, x: int) -> Node | None:
    """Unstable version: grow the list at the head or tail."""
    new_head = tail = head
    node = head
    while node:
        nxt = node.next
        if node.val < x:
            node.next = new_head
            new_head = node
        else:
            tail.next = node  # type: ignore[union-attr]
            tail = node
        node = nxt
    if tail:
        tail.next = None
    return new_head


for fn in (partition, partition_head_tail):
    out = to_list(fn(from_list([3, 5, 8, 5, 10, 2, 1]), 5))
    split = next(i for i, v in enumerate(out) if v >= 5)
    assert all(v < 5 for v in out[:split]) and all(v >= 5 for v in out[split:])
    assert sorted(out) == [1, 2, 3, 5, 5, 8, 10]
print("2.4 ok")

2.4 ok


## 2.5 Sum Lists
Two numbers are stored as linked lists of digits in **reverse** order (1's digit first). Return their sum as a list. Follow-up: digits stored in **forward** order.

In [6]:
def sum_lists(a: Node | None, b: Node | None) -> Node | None:
    dummy = tail = Node(0)
    carry = 0
    while a or b or carry:
        total = carry + (a.val if a else 0) + (b.val if b else 0)
        carry, digit = divmod(total, 10)
        tail.next = Node(digit)
        tail = tail.next
        a = a.next if a else None
        b = b.next if b else None
    return dummy.next


def sum_lists_forward(a: Node | None, b: Node | None) -> Node | None:
    """Pad the shorter with zeros, recurse to the end, and carry on the way back."""
    la, lb = to_list(a), to_list(b)
    width = max(len(la), len(lb))
    a = from_list([0] * (width - len(la)) + la)
    b = from_list([0] * (width - len(lb)) + lb)

    def helper(x: Node | None, y: Node | None) -> tuple[Node | None, int]:
        if x is None:
            return None, 0
        rest, carry = helper(x.next, y.next)  # type: ignore[union-attr]
        carry, digit = divmod(x.val + y.val + carry, 10)  # type: ignore[union-attr]
        return Node(digit, rest), carry

    head, carry = helper(a, b)
    return Node(carry, head) if carry else head


assert to_list(sum_lists(from_list([7, 1, 6]), from_list([5, 9, 2]))) == [2, 1, 9]  # 617 + 295 = 912
assert to_list(sum_lists(from_list([9, 9]), from_list([1]))) == [0, 0, 1]
assert to_list(sum_lists_forward(from_list([6, 1, 7]), from_list([2, 9, 5]))) == [9, 1, 2]
assert to_list(sum_lists_forward(from_list([9, 9]), from_list([1]))) == [1, 0, 0]
print("2.5 ok")

2.5 ok


## 2.6 Palindrome
Is a linked list a palindrome? Approach: runner finds the middle while pushing the first half onto a stack.

In [7]:
def is_palindrome(head: Node | None) -> bool:
    slow = fast = head
    stack: list[int] = []
    while fast and fast.next:
        stack.append(slow.val)  # type: ignore[union-attr]
        slow = slow.next  # type: ignore[union-attr]
        fast = fast.next.next
    if fast:  # odd length: skip middle
        slow = slow.next  # type: ignore[union-attr]
    while slow:
        if stack.pop() != slow.val:
            return False
        slow = slow.next
    return True


def is_palindrome_reverse(head: Node | None) -> bool:
    """Alternative: compare with a reversed copy."""
    values = to_list(head)
    return values == values[::-1]


for fn in (is_palindrome, is_palindrome_reverse):
    assert fn(from_list([0, 1, 2, 1, 0])) and fn(from_list([1, 2, 2, 1])) and fn(None)
    assert not fn(from_list([1, 2, 3]))
print("2.6 ok")

2.6 ok


## 2.7 Intersection
Two singly linked lists may share a tail (by reference, not by value). Return the first shared node. Trick: if they intersect, their last nodes are identical; chop the longer list's head so both have equal length, then walk together.

In [8]:
def length_and_tail(head: Node) -> tuple[int, Node]:
    n = 1
    while head.next:
        head = head.next
        n += 1
    return n, head


def intersection(a: Node | None, b: Node | None) -> Node | None:
    if a is None or b is None:
        return None
    len_a, tail_a = length_and_tail(a)
    len_b, tail_b = length_and_tail(b)
    if tail_a is not tail_b:
        return None
    if len_a < len_b:
        a, b, len_a, len_b = b, a, len_b, len_a
    for _ in range(len_a - len_b):
        a = a.next  # type: ignore[assignment]
    while a is not b:
        a, b = a.next, b.next  # type: ignore[assignment, union-attr]
    return a


def intersection_two_pointer(a: Node | None, b: Node | None) -> Node | None:
    """Elegant variant: each pointer walks A then B; they meet at the join (or both None)."""
    p, q = a, b
    while p is not q:
        p = p.next if p else b
        q = q.next if q else a
    return p


shared = from_list([7, 2, 1])
a = from_list([3, 1, 5, 9]); length_and_tail(a)[1].next = shared
b = from_list([4, 6]);       length_and_tail(b)[1].next = shared
for fn in (intersection, intersection_two_pointer):
    assert fn(a, b) is shared
    assert fn(from_list([1, 2]), from_list([1, 2])) is None
print("2.7 ok")

2.7 ok


## 2.8 Loop Detection
Return the node at the start of a cycle. Floyd's algorithm: fast (2 steps) and slow (1 step) meet inside the loop at a point `k` steps before the loop start (where `k` is the non-loop prefix length). Reset one pointer to the head and step both by 1 — they meet at the loop start.

In [9]:
def loop_start(head: Node | None) -> Node | None:
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next  # type: ignore[union-attr]
        if slow is fast:
            break
    else:
        return None
    slow = head
    while slow is not fast:
        slow, fast = slow.next, fast.next  # type: ignore[union-attr]
    return slow


lst = from_list(list(range(10)))
nodes = list(iter_nodes(lst))
nodes[-1].next = nodes[3]
assert loop_start(lst) is nodes[3]
assert loop_start(from_list([1, 2, 3])) is None
print("2.8 ok")

2.8 ok
